In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ============================================================
# 1. Setup
# ============================================================
os.environ["WANDB_DISABLED"] = "true"
import gdown
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding, EarlyStoppingCallback
import torch
from datasets import Dataset, Sequence, Value


2025-07-27 02:07:12.815986: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753582033.193242      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753582033.295208      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# ============================================================
# 2. Download Model and Dataset from Google Drive
# ============================================================

# Download model folder (containing all files)
#!gdown --folder https://drive.google.com/drive/folders/1bW-Ekjh7O1o91aa8Ro6UqTbHOihuL9bV
!gdown --folder https://drive.google.com/drive/folders/1G-f8Lu3aoMUxcnjyuZsoTRaNqAPMMihl

# Download the preprocessed CSV files (training and validation sets only)
!gdown https://drive.google.com/uc?id=1MbhRs35jrqf2pfCDOuK53tMtE9Q6hrtd
!gdown https://drive.google.com/uc?id=1nyL2H9pLhybbLQzbWUnJEGMwc7_aKw4b

# ============================================================
# 3. Load Data
# ============================================================

df_train = pd.read_csv("train_data.csv")
df_val = pd.read_csv("val_data.csv")

label_names = df_train.columns[1:]  # All emotion columns

texts_train = df_train['text'].tolist()
labels_train = df_train[label_names].values.tolist()
texts_val = df_val['text'].tolist()
labels_val = df_val[label_names].values.tolist()

# Create Hugging Face Dataset
train_ds = Dataset.from_dict({'text': texts_train, 'labels': labels_train})
eval_ds = Dataset.from_dict({'text': texts_val, 'labels': labels_val})

# ============================================================
# 4. Tokenizer and Model Loading
# ============================================================

tokenizer = DistilBertTokenizerFast.from_pretrained('./distilbert-base-uncased') # Name of model folder

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True)

train_ds = train_ds.map(tokenize_function, batched=True)
eval_ds = eval_ds.map(tokenize_function, batched=True)

# Convert labels to float to fix mismatch error
train_ds = train_ds.cast_column("labels", Sequence(Value("float32")))
eval_ds = eval_ds.cast_column("labels", Sequence(Value("float32")))

Retrieving folder contents
Processing file 1IVqe1yy9bhCmLlZC-PyjGZD6ePU4FliW config.json
Processing file 1Rn1OTP7idfzwzDQjndn9v9S07jCsxF13 model.safetensors
Processing file 1lE5DMSkV8yyZm3ddVqpmOgfRYJZapoVW special_tokens_map.json
Processing file 10lumW8CLsazofsEqCobWqGes2Gai1H8S tokenizer_config.json
Processing file 1JSLqB6faf99G6Xbx2uQcdqR0qAA18W9K tokenizer.json
Processing file 1gLk49KdVhNvjgkRL9OsLXLuYY38_Ifg4 vocab.txt
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1IVqe1yy9bhCmLlZC-PyjGZD6ePU4FliW
To: /kaggle/working/distilbert-base-uncased/config.json
100%|██████████████████████████████████████████| 581/581 [00:00<00:00, 3.36MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Rn1OTP7idfzwzDQjndn9v9S07jCsxF13
From (redirected): https://drive.google.com/uc?id=1Rn1OTP7idfzwzDQjndn9v9S07jCsxF13&confirm=t&uuid=29936f99-640f-422a-ba78-7349e7d2a0a5
To: /kaggle

Map:   0%|          | 0/186998 [00:00<?, ? examples/s]

Map:   0%|          | 0/10408 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/186998 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/10408 [00:00<?, ? examples/s]

In [4]:
# ============================================================
# 5. Model Init
# ============================================================

model = DistilBertForSequenceClassification.from_pretrained(
    './distilbert-base-uncased',
    num_labels=len(label_names),
    problem_type="multi_label_classification"
)

# ============================================================
# 6. Training
# ============================================================

training_args = TrainingArguments(
    output_dir="./emotion_model",
    eval_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=50,
    learning_rate=5e-5,
    weight_decay=0.1,
    save_strategy="epoch",
    save_total_limit=3,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)
trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at ./distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.113100,0.112189
2,0.109100,0.110998
3,0.107200,0.112724
4,0.100700,0.113913
5,0.097900,0.115936


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=14610, training_loss=0.10691200520876415, metrics={'train_runtime': 2576.4045, 'train_samples_per_second': 3629.05, 'train_steps_per_second': 56.707, 'total_flos': 9272260873491648.0, 'train_loss': 0.10691200520876415, 'epoch': 5.0})

In [5]:
# ============================================================
# 7. Save Final Model
# ============================================================

model.save_pretrained("./trained_emotion_model")
tokenizer.save_pretrained("./trained_emotion_model")

('./trained_emotion_model/tokenizer_config.json',
 './trained_emotion_model/special_tokens_map.json',
 './trained_emotion_model/vocab.txt',
 './trained_emotion_model/added_tokens.json',
 './trained_emotion_model/tokenizer.json')